# Part 1: Python Core

This notebook is a practical reference for Python's language fundamentals, core data structures, and important behavior rules.

## How to use this notebook

1. Read the explanation immediately before each code cell.
2. Run the code, inspect the output, and change one thing at a time.
3. Complete the **Try it** prompts without looking at a solution first.
4. Remember: a variable is a name bound to an object; it is not a box containing a value.

## Learning goals

By the end of this part you should be able to:

- Predict how names, objects, identity, equality, and mutability interact.
- Write functions using positional and keyword arguments, `*args`, and `**kwargs`.
- Explain scope using namespaces and the LEGB rule.
- Use comprehensions, iterators, generators, and closures intentionally.
- Select an appropriate built-in data structure and estimate its time and memory costs.
- Explain copying, hashing, late binding, dictionary/set lookup, list resizing, and string immutability.

## 1. Language fundamentals

### Variables, references, and the object model

Python evaluates an assignment by binding a name to an object:

```python
name = object
```

Every object has an identity (`id`), a type (`type`), and a value. Names live in namespaces; objects live independently of the names that reference them. Rebinding a name changes what it refers to, while mutation changes an object in place.

- `a = b` creates another reference; it does not copy `b`.
- `a == b` asks whether values compare equal.
- `a is b` asks whether both names refer to the same object. Use `is` primarily for singleton checks such as `value is None`.
- `None` is Python's singleton object meaning "no value" or "not supplied". It is falsy, but it is not the same as `False`, `0`, or `""`.

### Mutable and immutable objects

Immutable examples include `int`, `float`, `bool`, `str`, `bytes`, and `tuple` (provided its members are not mutated). Mutable examples include `list`, `dict`, `set`, and `bytearray`.

An immutable object's apparent change creates a new object. A mutable object's methods can change the existing object. A tuple can be immutable while containing a mutable list, so immutability is not always recursively deep.

### Truthiness

`False`, `None`, numeric zero, empty strings, and empty containers are falsy. Most other objects are truthy. Custom classes can define truthiness with `__bool__`; if absent, Python uses `__len__`.

Prefer `if items:` when you mean "has any items". Use `if items is None:` when you must distinguish an absent value from an empty value.

In [1]:
# References, identity, equality, mutability, and truthiness
first = [1, 2]
second = first
third = [1, 2]

print("second is first:", second is first)
print("third == first:", third == first)
print("third is first:", third is first)

second.append(3)
print("first after mutating second:", first)

number = 10
original_id = id(number)
number += 1
print("rebinding changed object identity:", id(number) != original_id)

empty_values = [None, False, 0, 0.0, "", [], {}, set()]
print("falsy values:", [value for value in empty_values if not value])

missing = None
print("None check:", missing is None)
print("None is False:", missing is False)

# A tuple is immutable as a container, but can refer to a mutable object.
record = ("Ada", ["Python"])
record[1].append("SQL")
print("mutable member inside tuple:", record)

second is first: True
third == first: True
third is first: False
first after mutating second: [1, 2, 3]
rebinding changed object identity: True
falsy values: [None, False, 0, 0.0, '', [], {}, set()]
None check: True
None is False: False
mutable member inside tuple: ('Ada', ['Python', 'SQL'])


### Functions, arguments, scope, and namespaces

Functions are first-class objects: they can be stored in variables, passed to other functions, returned, and placed in collections. `lambda` creates a small anonymous function and should remain readable; use `def` for reusable or documented logic.

A function call can combine:

- positional arguments, matched by position;
- keyword arguments, matched by parameter name;
- `*args`, which collects extra positional arguments into a tuple;
- `**kwargs`, which collects extra keyword arguments into a dictionary.

Python uses **argument passing by object reference** (also described as call-by-sharing). The function receives references to the same objects. Reassigning a parameter only changes the local name; mutating a shared mutable object is visible to the caller.

A namespace maps names to objects. The LEGB lookup order is **Local**, **Enclosing**, **Global**, **Built-in**. `global` makes assignments target a module-level name; `nonlocal` makes assignments target a name in an enclosing function. Use both sparingly because explicit data flow is usually easier to test.

A closure is an inner function that remembers values from its enclosing scope after the outer function returns. Closures are useful for configuration and state, but loops can expose the late-binding rule: a closure captures a variable, not its value at creation time. Default arguments can freeze the current loop value.

### Comprehensions and lazy iteration

A comprehension creates a collection compactly:

- list: `[expression for item in iterable if condition]`
- set: `{expression for item in iterable}`
- dict: `{key: value for item in iterable}`
- generator expression: `(expression for item in iterable)`

An **iterable** can produce an iterator, usually through `__iter__`. An **iterator** represents a stream of values and supplies `__next__`; it is consumed as you call `next`. A generator is a convenient iterator created by a function containing `yield`. `yield` pauses execution and preserves local state until the next request. Generator expressions provide the same lazy behavior in a compact form.

Lazy iteration uses little memory for large streams, but values are available only once unless you create a new iterator or materialize them.

In [ ]:
# Argument passing, first-class functions, closures, comprehensions, and generators

def summarize(label, *values, scale=1, **metadata):
    """Return a small report showing all argument categories."""
    return {
        "label": label,
        "values": tuple(value * scale for value in values),
        "metadata": metadata,
    }

print(summarize("scores", 2, 3, 4, scale=10, source="quiz"))

items = [1, 2, 3]

def rebind(values):
    values = ["new local list"]

def mutate(values):
    values.append(4)

rebind(items)
print("after rebind:", items)
mutate(items)
print("after mutation:", items)

# Functions can be passed and returned.
def apply(operation, value):
    return operation(value)

square = lambda value: value * value
print("first-class function:", apply(square, 5))

# A closure remembers the enclosing variable.
def make_counter(start=0):
    count = start

    def increment(step=1):
        nonlocal count
        count += step
        return count

    return increment

counter = make_counter(10)
print("closure state:", counter(), counter(2))

# Late binding: all functions read the final value of factor.
late = [lambda value: value * factor for factor in (1, 2, 3)]
fixed = [lambda value, factor=factor: value * factor for factor in (1, 2, 3)]
print("late binding:", [function(10) for function in late])
print("frozen default:", [function(10) for function in fixed])

numbers = range(1, 8)
print("list comprehension:", [number * number for number in numbers if number % 2])
print("set comprehension:", {number % 3 for number in numbers})
print("dict comprehension:", {number: number ** 2 for number in numbers})

iterator = iter([10, 20])
print("iterable produces iterator:", next(iterator), next(iterator))


def countdown(start):
    while start:
        yield start
        start -= 1

stream = countdown(3)
print("generator type:", type(stream).__name__)
print("generator values:", list(stream))

# This generator expression does not allocate a list of one million squares.
lazy_squares = (number * number for number in range(1_000_000))
print("generator expression sample:", next(lazy_squares), next(lazy_squares))

{'label': 'scores', 'values': (20, 30, 40), 'metadata': {'source': 'quiz'}}
after rebind: [1, 2, 3]
after mutation: [1, 2, 3, 4]
first-class function: 25
closure state: 11 13
late binding: [30, 30, 30]
frozen default: [10, 20, 30]
list comprehension: [1, 9, 25, 49]
set comprehension: {0, 1, 2}
dict comprehension: {1: 1, 2: 4, 3: 9, 4: 16, 5: 25, 6: 36, 7: 49}
iterable produces iterator: 10 20
generator type: generator
generator values: [3, 2, 1]
generator expression sample: 0 1


## 2. Core data structures

| Structure | Ordered? | Mutable? | Typical lookup / membership | Best use |
|---|---:|---:|---|---|
| `list` | Yes | Yes | index $O(1)$; membership $O(n)$ | General sequence, append-heavy work |
| `tuple` | Yes | No | index $O(1)$; membership $O(n)$ | Fixed records, hashable composite keys |
| `set` | No meaningful order | Yes | average membership $O(1)$ | Unique values and fast membership |
| `dict` | Insertion order | Yes | average key lookup $O(1)$ | Key-value mapping |
| `deque` | Yes | Yes | append/pop at either end $O(1)$ | Queues and sliding windows |
| `heapq` | List-backed min-heap | Yes | push/pop-min $O(\log n)$; peek-min $O(1)$ | Priority queues |
| `defaultdict` | Dict behavior | Yes | average key lookup $O(1)$ | Grouping and accumulating without prechecks |
| `Counter` | Dict-like counts | Yes | average key lookup $O(1)$ | Frequencies and most-common values |
| `OrderedDict` | Insertion order | Yes | average key lookup $O(1)$ | Order-sensitive APIs and move-to-end operations |
| `dataclass` | Class-defined fields | Usually mutable | Attribute access is typically $O(1)$ | Named records with generated methods |

### Memory and implementation notes

- A `list` is a dynamic array: it stores references in contiguous slots and over-allocates capacity when it grows. Appends are amortized $O(1)$, but inserting or deleting near the front shifts references and costs $O(n)$.
- A `tuple` is a compact fixed-size sequence of references. It cannot resize, so it generally uses less structural memory than an equivalent list.
- `dict` and `set` use hash tables. They rely on hashable keys/elements and resize to preserve efficient average lookup. Collisions are handled internally; order should not be treated as sorted order.
- `deque` is optimized for both ends, unlike a list whose left-end operations shift all later elements.
- `heapq` maintains the smallest item at index `0`; it is not a fully sorted list. Calling `sorted` is appropriate when you need all items ordered.
- `defaultdict`, `Counter`, and `OrderedDict` are specialized dictionary variants. Use a normal `dict` when their extra behavior is not needed.
- A `dataclass` is a class-generation helper, not a storage primitive. It makes records clearer by generating `__init__`, `__repr__`, and comparisons according to options.

### Choosing a structure

Use a `list` for a sequence you index or traverse; a `tuple` for a fixed record; a `set` when uniqueness or membership matters; a `dict` when values are identified by keys; a `deque` for FIFO/LIFO operations at both ends; a heap for repeatedly retrieving the smallest priority; and a dataclass when domain data has named fields and behavior may grow.

In [3]:
from collections import Counter, OrderedDict, defaultdict, deque
from dataclasses import dataclass
import heapq

numbers = [4, 1, 4, 2]
fixed_record = ("Ada", "Python")
unique_numbers = set(numbers)
profile = {"name": "Ada", "topics": 3}

print("list:", numbers, "index:", numbers[1])
print("tuple:", fixed_record)
print("set membership:", 2 in unique_numbers, "unique:", unique_numbers)
print("dict lookup:", profile["name"])

queue = deque(["task-1", "task-2"])
queue.append("task-3")
print("deque FIFO:", queue.popleft(), list(queue))

priorities = []
for priority, task in [(3, "email"), (1, "backup"), (2, "report")]:
    heapq.heappush(priorities, (priority, task))
print("heap order:", [heapq.heappop(priorities) for _ in range(len(priorities))])

groups = defaultdict(list)
for language, person in [("Python", "Ada"), ("Python", "Guido"), ("C", "Dennis")]:
    groups[language].append(person)
print("defaultdict grouping:", dict(groups))

counts = Counter("mississippi")
print("Counter:", counts)
print("most common:", counts.most_common(2))

ordered = OrderedDict([("first", 1), ("second", 2)])
ordered.move_to_end("first")
print("OrderedDict after move:", ordered)

@dataclass
class User:
    name: str
    visits: int = 0

user = User("Ada")
user.visits += 1
print("dataclass:", user)

# Compare membership operations with a small, reproducible example.
print("list membership scans values:", 99 in numbers)
print("set membership uses hashing:", 99 in unique_numbers)

list: [4, 1, 4, 2] index: 1
tuple: ('Ada', 'Python')
set membership: True unique: {1, 2, 4}
dict lookup: Ada
deque FIFO: task-1 ['task-2', 'task-3']
heap order: [(1, 'backup'), (2, 'report'), (3, 'email')]
defaultdict grouping: {'Python': ['Ada', 'Guido'], 'C': ['Dennis']}
Counter: Counter({'i': 4, 's': 4, 'p': 2, 'm': 1})
most common: [('i', 4), ('s', 4)]
OrderedDict after move: OrderedDict({'second': 2, 'first': 1})
dataclass: User(name='Ada', visits=1)
list membership scans values: False
set membership uses hashing: False


## 3. Important Python behavior

### Assignment, copying, and argument passing

Assignment binds another name to the same object. A shallow copy creates a new outer container but keeps references to nested objects. A deep copy recursively copies nested objects (subject to custom behavior and special cases).

Use `copy.copy` when the outer container should be independent but nested objects may remain shared. Use `copy.deepcopy` when nested mutable state must also be independent. Prefer explicit construction or immutable data when that makes ownership clearer.

### Mutable default arguments

Default parameter expressions are evaluated once, when the `def` statement executes, not each time the function is called. Therefore, a mutable default such as `items=[]` is shared across calls. The standard pattern is `items=None`, followed by creating a new list inside the function.

### Hashability and dictionary hashing

A hashable object has a stable hash for its lifetime and equality compatible with that hash: if `a == b`, then `hash(a) == hash(b)`. Immutable built-ins such as strings, numbers, and tuples of hashable values can be hashable. Lists, dictionaries, and sets are unhashable because mutation could invalidate their table position.

Dictionaries and sets use hash tables. Python hashes a key or element, maps the hash to a table position, and resolves collisions internally. Lookup is average-case $O(1)$ but can degrade in pathological collision scenarios. Dictionary keys must be unique; assigning an equal key replaces the existing value. Modern Python dictionaries preserve insertion order, but that does not make them sorted.

### List resizing and string immutability

Lists store references in a dynamically resized array. To avoid reallocating on every append, Python usually allocates spare capacity. Most appends are therefore amortized $O(1)$, while front insertion/removal is $O(n)$ because references shift.

Strings are immutable sequences of Unicode characters. Operations that appear to modify a string create a new string, so repeated concatenation in a large loop can do unnecessary work. Accumulate pieces in a list and call `"".join(pieces)` when building substantial text.

### Equality, identity, and special methods

`==` can be customized through `__eq__`; `is` is identity and cannot be overloaded. Classes can also define `__hash__`, `__bool__`, `__len__`, and ordering methods. If equality and hashing are customized, preserve the invariant that equal objects have equal hashes.

In [5]:
import copy

# Assignment shares both the outer object and nested objects.
source = [[1], [2]]
alias = source
shallow = copy.copy(source)
deep = copy.deepcopy(source)

source[0].append(99)
source.append([3])
print("alias sees all changes:", alias)
print("shallow shares nested lists but not outer list:", shallow)
print("deep is independent recursively:", deep)

# The safe pattern for mutable defaults.
def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items

print("fresh default call 1:", add_item("a"))
print("fresh default call 2:", add_item("b"))

# This intentionally demonstrates the surprising behavior.
def unsafe_add_item(item, items=[]):
    items.append(item)
    return items

print("shared default call 1:", unsafe_add_item("a"))
print("shared default call 2:", unsafe_add_item("b"))

print("hashable values:", hash("python"), hash(("python", 3)))
try:
    hash(["python"])
except TypeError as error:
    print("unhashable list:", error)

class Score:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        return isinstance(other, Score) and self.value == other.value

left = Score(10)
right = Score(10)
print("custom equality:", left == right)
print("different identity:", left is right)

parts = ["Python", "is", "readable"]
joined = " ".join(parts)
print("joined immutable string:", joined)
print("original parts remain:", parts)

alias sees all changes: [[1, 99], [2], [3]]
shallow shares nested lists but not outer list: [[1, 99], [2]]
deep is independent recursively: [[1], [2]]
fresh default call 1: ['a']
fresh default call 2: ['b']
shared default call 1: ['a']
shared default call 2: ['a', 'b']
hashable values: -6795461659622274306 510585929178991489
unhashable list: unhashable type: 'list'
custom equality: True
different identity: False
joined immutable string: Python is readable
original parts remain: ['Python', 'is', 'readable']


## 4. Practice lab

Work through these without changing the earlier examples.

1. Write `describe(value)` so that it reports `"missing"` only for `None`, and `"empty"` for an empty container.
2. Write `partition(*values)` that returns a dictionary with even and odd values.
3. Rewrite a nested loop as a dictionary comprehension, then explain its time and memory cost.
4. Create a generator that yields only lines beginning with `ERROR` from an iterable of strings.
5. Implement a queue twice: once with a list and once with `deque`. Explain why repeated `pop(0)` is slower.
6. Build a word-frequency `Counter`, then return the three most common words.
7. Create a frozen dataclass and test whether instances can be used as dictionary keys.
8. Predict the output before running the late-binding example above, then fix it with a default argument and with a closure factory.
9. Explain why `(1, [2])` cannot be hashed even though the tuple itself is immutable.
10. Use `timeit` to compare membership in a list and a set for a value near the end of the collection.

### Self-check checklist

- [ ] I can explain name binding versus object mutation.
- [ ] I use `is None` rather than `== None`.
- [ ] I can choose between positional and keyword arguments.
- [ ] I understand `*args` and `**kwargs` at both call sites and function definitions.
- [ ] I can trace a name using LEGB and distinguish `global` from `nonlocal`.
- [ ] I know when a generator saves memory and when materializing a list is simpler.
- [ ] I know average versus worst-case lookup complexity for hash tables.
- [ ] I can explain shallow versus deep copying and avoid mutable defaults.
- [ ] I can predict late binding and string/list mutation behavior.

### Quick reference

- **Need order and indexing?** `list` or `tuple`.
- **Need uniqueness or membership?** `set`.
- **Need key-value lookup?** `dict`.
- **Need both-end queue operations?** `deque`.
- **Need repeated minimum priority?** `heapq`.
- **Need grouping/counting?** `defaultdict` or `Counter`.
- **Need a named domain record?** `dataclass`.
- **Need one-pass, low-memory processing?** iterator or generator.